# Feature attributions for Integrated Gradients

Plot IG attributions on LAAT and CAML and find standard deviations for random attributions

In [ ]:
from captum.attr import configure_interpretable_embedding_layer
from captum.attr import remove_interpretable_embedding_layer
from matplotlib import pyplot as plt
import torch
import numpy as np
import random

import config
import loader
import attributor
import evaluator

SCOPES = config.SCOPES
SUB_BITS = config.SUB_BITS
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
METHOD_NAME = 'ig'
N_STEPS = 200

## LAAT

In [ ]:
# Load model
model_name = 'laat'
model, dataloader = loader.load_model_and_data(model_name)
def model_wrapper(*args, **kwargs):
    output, _ = model(*args, **kwargs)
    return torch.sigmoid(output[1])
int_emb = configure_interpretable_embedding_layer(model, 'embedding')
model.train()

In [ ]:
# Create IG attribution methods
methods = {}
for scope in SCOPES:
    methods[scope] = attributor.create_method(model_wrapper, scope, METHOD_NAME)

In [ ]:
predss = {}
attrs = {}
N = 3
for idx, tup in enumerate(dataloader):
    # Attribute only the N first samples of the subset
    if SUB_BITS[idx] == 0 or np.count_nonzero(SUB_BITS[:idx]) > N-1:
        continue
    print("Attributing sample", idx)
    predss[idx] = {}
    attrs[idx] = {}
    input_embed, base_embed, afa = evaluator.prepare_input(model_name, tup, int_emb)
    preds = model_wrapper(input_embed, afa)[0]
    for target_idx, pred in enumerate(preds):
        if pred.item() > config.THRESH:
            # Attribute only 20% to save time
            if random.uniform(0, 1) > 0.2:
                continue
            print("Attributing target_idx", target_idx)
            predss[idx][target_idx] = pred.item()
            attrs[idx][target_idx] = {}
            # Compute attributions
            for scope in SCOPES:
                a = attributor.attribute(scope,
                                         METHOD_NAME,
                                         methods[scope],
                                         input_embed,
                                         base_embed,
                                         afa,
                                         target_idx,
                                         n_steps=N_STEPS,
                                         n_samples=None)
                attrs[idx][target_idx][scope] = a

In [ ]:
# Find standard deviation for local and global random attributions
# Take std of IG on first sample, first target
for idx, preds in predss.items():
    for target_idx, pred in preds.items():
        std_local = torch.std(attrs[idx][target_idx]['local'], unbiased=False).item()
        std_global = torch.std(attrs[idx][target_idx]['global'], unbiased=False).item()
        break
    break
std_local = round(std_local, 6)
std_global = round(std_global, 6)
print("std_local:", std_local)
print("std_global:", std_global)

In [ ]:
# Plot attributions
for idx, preds in predss.items():
    for target_idx, pred in preds.items():
        print("sample_idx:", idx)
        print("target_idx:", target_idx)
        print("pred:", round(pred, 2))
        for scope, a in attrs[idx][target_idx].items():
            a = a.sum(dim=2).squeeze(0).cpu().detach().numpy()
            plt.bar(range(len(a)), a)
            plt.show()

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)

## CAML

In [ ]:
# Load model
model_name = 'caml'
model, dataloader = loader.load_model_and_data(model_name)
def model_wrapper(*args, **kwargs):
    output, _, _ = model(*args, **kwargs)
    return torch.sigmoid(output)
int_emb = configure_interpretable_embedding_layer(model, 'embed')
model.train()

In [ ]:
# Create IG attribution methods
methods = {}
for scope in SCOPES:
    methods[scope] = attributor.create_method(model_wrapper, scope, METHOD_NAME)

In [ ]:
predss = {}
attrs = {}
N = 3
for idx, tup in enumerate(dataloader):
    # Attribute only the N first samples of the subset
    if SUB_BITS[idx] == 0 or np.count_nonzero(SUB_BITS[:idx]) > N-1:
        continue
    print("Attributing sample", idx)
    predss[idx] = {}
    attrs[idx] = {}
    input_embed, base_embed, afa = evaluator.prepare_input(model_name, tup, int_emb)
    preds = model_wrapper(input_embed, afa)[0]
    for target_idx, pred in enumerate(preds):
        if pred.item() > config.THRESH:
            # Attribute only 20% to save time
            if random.uniform(0, 1) > 0.2:
                continue
            print("Attributing target_idx", target_idx)
            predss[idx][target_idx] = pred.item()
            attrs[idx][target_idx] = {}
            # Compute attributions
            for scope in config.SCOPES:
                a = attributor.attribute(scope,
                                         METHOD_NAME,
                                         methods[scope],
                                         input_embed,
                                         base_embed,
                                         afa,
                                         target_idx,
                                         n_steps=N_STEPS,
                                         n_samples=None)
                attrs[idx][target_idx][scope] = a

In [ ]:
# Find standard deviation for local and global random attributions
# Take std of IG on first sample, first target
for idx, preds in predss.items():
    for target_idx, pred in preds.items():
        std_local = torch.std(attrs[idx][target_idx]['local'], unbiased=False).item()
        std_global = torch.std(attrs[idx][target_idx]['global'], unbiased=False).item()
        break
    break
std_local = round(std_local, 6)
std_global = round(std_global, 6)
print("std_local:", std_local)
print("std_global:", std_global)

In [ ]:
# Plot attributions
for idx, preds in predss.items():
    for target_idx, pred in preds.items():
        print("sample_idx:", idx)
        print("target_idx:", target_idx)
        print("pred:", round(pred, 2))
        for scope, a in attrs[idx][target_idx].items():
            a = a.sum(dim=2).squeeze(0).cpu().detach().numpy()
            plt.bar(range(len(a)), a)
            plt.show()

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)

## Evaluate attributions using max_sensitivity

In [ ]:
from captum.metrics import sensitivity_max

In [ ]:
int_emb = configure_interpretable_embedding_layer(model, 'embed')
model = model.half()
model.train()
methods = attributor.create_methods(model_wrapper)

In [ ]:
for idx, tup in enumerate(dataloader):
    # Evaluate only the first sample of the subset
    if SUB_BITS[idx] == 0 or np.count_nonzero(SUB_BITS[:idx]) > 1:
        continue
    print("Attributing sample", idx)

    # Prepare input and baseline
    input_indices, afa, _, _, _ = tup
    input_indices = torch.LongTensor(input_indices).to(DEVICE)
    afa = torch.FloatTensor(afa).to(DEVICE)
    afa = afa.type(torch.half)
    input_embed = int_emb.indices_to_embeddings(input_indices).to(DEVICE)
    input_embed = input_embed.type(torch.half)
    base_embed = torch.zeros_like(input_embed).to(DEVICE)
    base_embed = base_embed.type(torch.half)

    preds = model_wrapper(input_embed, afa)[0]

    # Evaluate sample
    for target_idx, pred in enumerate(preds):
        # Evaluate only positive predictions
        if pred.item() > config.THRESHOLD:
            method = methods['global']['ig']
            # Compute maxsen scores
            print("Evaluating target_idx:", target_idx)
            maxsen = sensitivity_max(method.attribute,
                                     input_embed,
                                     n_perturb_samples=10,
                                     baselines=base_embed,
                                     target=target_idx,
                                     additional_forward_args=afa)
            maxsen = maxsen.cpu().item()
            print(maxsen)

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)